# 본 분석 전 기술통계 및 전처리 필요 항목 점검

본 분석에선 실제 전처리를 수행하지 않고,\
**분석에 들어가기 앞서 어떤 전처리가 필요한지 확인하는 진단용 기술통계**입니다.

## 확인 항목
| 기준 | 확인 내용 |
|---|---|
| 결측 | 컬럼별 결측치 수와 비율 |
| 중복 | 완전 중복 행, 주요 키 중복 |
| 공백 | 문자열 컬럼의 빈 문자열, 앞뒤 공백, 연속 공백 |
| 타입 | 컬럼별 dtype, 고유값 수, 구조형 문자열 여부 |
| 이상치 | 수치형 컬럼의 IQR 기준 이상치 후보 |
| 조인 가능성 | `appid` 기준 테이블 간 매칭률 |

## 점검 범위
- `steam_indie_games.csv`
- `steam_indie_tags.csv`
- `steam_indie_review_summary.csv`
- `steam_indie_review_histogram.csv`
- `steam_indie_reviews.csv`
- `steam_indie_stratified_sample.csv`가 있으면 추가 확인

> 주의: 여기서는 컬럼 삭제, 결측치 대체, 이상치 제거 같은 전처리는 하지 않습니다.  
> 목적은 전처리가 필요한 지점을 찾는 것입니다.

# 1. 라이브러리 호출

In [1]:
from pathlib import Path
import ast
import json
import re
import platform

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# 한글 폰트 설정
import platform
if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':  # macOS
    plt.rcParams[
        'font.family'] = 'AppleGothic'
else:  # Linux
    plt.rcParams['font.family'] = 'NanumGothic'

plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (12, 6)

# 보기 옵션
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 160)

# 2. 파일 경로 설정 및 데이터 로드

In [2]:
# 프로젝트 루트 직접 지정
# 다른 환경에서 실행할 경우 ROOT만 본인 프로젝트 경로에 맞게 수정
# db에서 바로 불러오는법을 모름
ROOT = Path(r"C:\Users\joon5\Documents\github\steam-indie-game-analysis")

# 원천/소스 파일이 들어있는 폴더
DATA_DIR = ROOT / "data" / "processed"

# 위 경로가 없으면 현재 노트북이 있는 폴더 기준으로 읽도록 보조 처리
# Colab, Kaggle, 업로드 파일 테스트 환경에서만 사용
if not DATA_DIR.exists():
    DATA_DIR = Path(".")

# 파일 경로
GAMES_PATH = DATA_DIR / "steam_indie_games.csv"
TAGS_PATH = DATA_DIR / "steam_indie_tags.csv"
REVIEW_SUMMARY_PATH = DATA_DIR / "steam_indie_review_summary.csv"
REVIEW_HISTOGRAM_PATH = DATA_DIR / "steam_indie_review_histogram.csv"
REVIEWS_PATH = DATA_DIR / "steam_indie_reviews.csv"


# 경로 확인
# 파일이 없으면 이후 read_csv 단계에서 에러가 나므로, 먼저 exists() 결과를 확인한다.
print("ROOT                   =", ROOT)
print("DATA_DIR               =", DATA_DIR)
print("GAMES_PATH             =", GAMES_PATH)
print("TAGS_PATH              =", TAGS_PATH)
print("REVIEW_SUMMARY_PATH    =", REVIEW_SUMMARY_PATH)
print("REVIEW_HISTOGRAM_PATH  =", REVIEW_HISTOGRAM_PATH)
print("REVIEWS_PATH           =", REVIEWS_PATH)

print()
print("games exists             :", GAMES_PATH.exists())
print("tags exists              :", TAGS_PATH.exists())
print("review_summary exists    :", REVIEW_SUMMARY_PATH.exists())
print("review_histogram exists  :", REVIEW_HISTOGRAM_PATH.exists())
print("reviews exists           :", REVIEWS_PATH.exists())

# 데이터 읽기
# 여기서는 원본을 바로 전처리하지 않고, 파일을 읽어온 뒤 기술통계만 확인한다.
games_df = pd.read_csv(GAMES_PATH)
tags_df = pd.read_csv(TAGS_PATH)
review_summary_df = pd.read_csv(REVIEW_SUMMARY_PATH)
review_histogram_df = pd.read_csv(REVIEW_HISTOGRAM_PATH)
reviews_df = pd.read_csv(REVIEWS_PATH, low_memory=False)


ROOT                   = C:\Users\joon5\Documents\github\steam-indie-game-analysis
DATA_DIR               = C:\Users\joon5\Documents\github\steam-indie-game-analysis\data\processed
GAMES_PATH             = C:\Users\joon5\Documents\github\steam-indie-game-analysis\data\processed\steam_indie_games.csv
TAGS_PATH              = C:\Users\joon5\Documents\github\steam-indie-game-analysis\data\processed\steam_indie_tags.csv
REVIEW_SUMMARY_PATH    = C:\Users\joon5\Documents\github\steam-indie-game-analysis\data\processed\steam_indie_review_summary.csv
REVIEW_HISTOGRAM_PATH  = C:\Users\joon5\Documents\github\steam-indie-game-analysis\data\processed\steam_indie_review_histogram.csv
REVIEWS_PATH           = C:\Users\joon5\Documents\github\steam-indie-game-analysis\data\processed\steam_indie_reviews.csv

games exists             : True
tags exists              : True
review_summary exists    : True
review_histogram exists  : True
reviews exists           : True


In [3]:
# 크기 확인
print()
print("games_df shape            :", games_df.shape)
print("tags_df shape             :", tags_df.shape)
print("review_summary_df shape   :", review_summary_df.shape)
print("review_histogram_df shape :", review_histogram_df.shape)
print("reviews_df shape          :", reviews_df.shape)


games_df shape            : (9692, 14)
tags_df shape             : (9706, 10)
review_summary_df shape   : (200, 7)
review_histogram_df shape : (11782, 10)
reviews_df shape          : (236379, 21)


In [4]:
# 컬럼 목록 확인
print('[games_df columns]')
print(games_df.columns.tolist())
print()

print('[tags_df columns]')
print(tags_df.columns.tolist())
print()

print('[review_summary_df columns]')
print(review_summary_df.columns.tolist())
print()

print('[review_histogram_df columns]')
print(review_histogram_df.columns.tolist())
print()

print('[reviews_df columns]')
print(reviews_df.columns.tolist())

[games_df columns]
['appid', 'owners', 'positive', 'negative', 'price', 'ccu', 'genres', 'release_date', 'developers', 'total_reviews', 'owners_lower', 'is_f2p', 'is_early_access', 'name']

[tags_df columns]
['appid', 'name', 'developer', 'publisher', 'owners', 'positive', 'negative', 'price', 'tags', 'updated_at']

[review_summary_df columns]
['appid', 'review_score', 'review_score_desc', 'total_positive', 'total_negative', 'total_reviews', 'collected_at']

[review_histogram_df columns]
['appid', 'name', 'stratum', 'release_date', 'hist_start_date', 'hist_end_date', 'date', 'recommendations_up', 'recommendations_down', 'data_type']

[reviews_df columns]
['recommendationid', 'appid', 'language', 'review', 'timestamp_created', 'timestamp_updated', 'voted_up', 'votes_up', 'votes_funny', 'weighted_vote_score', 'comment_count', 'steam_purchase', 'received_for_free', 'written_during_early_access', 'author_steamid', 'author_num_games_owned', 'author_num_reviews', 'author_playtime_forever', '

# 3. 점검용 함수 정의

In [5]:
def check_basic_info(df, df_name, exclude_cols=None):
    """행/열 수, 완전 중복 행, 컬럼별 타입/결측/고유값을 한 번에 확인"""
    print(f"\n{'='*80}")
    print(f"{df_name}의 기본 정보 / 타입 / 결측치 확인")
    print(f"{'='*80}\n")

    # 제외할 컬럼 반영
    df_copied = df.copy()
    if exclude_cols:
        df_copied = df_copied.drop(columns=exclude_cols, errors='ignore')

    # dict, list, set 같은 해시 불가능 값이 들어있는 컬럼은 문자열로 변환
    for col in df_copied.columns:
        try:
            df_copied[col].nunique(dropna=True)
        except TypeError:
            df_copied[col] = df_copied[col].astype(str)

    # 전체 요약
    overview_df = pd.DataFrame({
        '항목': ['행 개수', '열 개수', '중복 행 개수'],
        '값': [df_copied.shape[0], df_copied.shape[1], df_copied.duplicated().sum()]
    })

    summary_df = pd.DataFrame({
        '데이터타입': df_copied.dtypes.astype(str),
        '행 개수': df_copied.count(),
        '행 비율(%)': (df_copied.count() / len(df_copied) * 100).round(2),
        '결측치 개수': df_copied.isnull().sum(),
        '결측치 비율(%)': (df_copied.isnull().sum() / len(df_copied) * 100).round(2),
        '고유값 개수': df_copied.nunique(dropna=True)
    }).sort_values(by=['결측치 개수', '고유값 개수'], ascending=[False, False])

    print("[전체 요약]")
    display(overview_df)

    print("[컬럼별 요약]")
    display(summary_df)

    print("[상위 5행]")
    display(df_copied.head())

In [6]:
def check_id_duplicates(df, col_name, df_name, top_n=10):
    """기준 키로 쓸 컬럼의 중복 여부를 확인"""
    print(f"\n{'='*80}")
    print(f"{df_name}의 {col_name} 값 중복 확인")
    print(f"{'='*80}")

    df_copied = df.copy()

    if col_name not in df_copied.columns:
        print(f"'{col_name}' 컬럼이 존재하지 않습니다.")
        return

    duplicate_count = df_copied[col_name].duplicated().sum()

    print('전체 행 수:', len(df_copied))
    print(f'{col_name} 고유 개수:', df_copied[col_name].nunique(dropna=True))
    print(f'중복 {col_name} 개수:', duplicate_count)

    if duplicate_count > 0:
        print()
        print('[중복 상위 값]')
        dup_summary = df_copied[col_name].value_counts(dropna=False).reset_index()
        dup_summary.columns = [col_name, '등장 횟수']
        display(dup_summary[dup_summary['등장 횟수'] > 1].head(top_n))
    else:
        print('중복 값이 없습니다.')

In [7]:
def check_string_space_summary(df, df_name, top_n=30):
    """문자열 컬럼의 빈 문자열, 앞뒤 공백, 연속 공백을 확인"""
    print(f"\n{'='*80}")
    print(f"{df_name}의 문자열 공백/빈값 확인")
    print(f"{'='*80}")

    object_cols = df.select_dtypes(include=["object", "string"]).columns.tolist()

    if len(object_cols) == 0:
        print("문자열 컬럼이 없습니다.")
        return

    rows = []

    for col in object_cols:
        temp = df[col]
        temp_str = temp.dropna().astype(str)

        empty_count = temp_str.str.strip().eq("").sum()
        leading_trailing_count = temp_str.ne(temp_str.str.strip()).sum()
        multi_space_count = temp_str.str.contains(r"\s{2,}", regex=True).sum()

        rows.append({
            "컬럼명": col,
            "문자열 행 수": len(temp_str),
            "빈 문자열/공백값 개수": empty_count,
            "앞뒤 공백 개수": leading_trailing_count,
            "연속 공백 포함 개수": multi_space_count
        })

    summary_df = pd.DataFrame(rows)
    summary_df = summary_df.sort_values(
        by=["빈 문자열/공백값 개수", "앞뒤 공백 개수", "연속 공백 포함 개수"],
        ascending=False
    )

    display(summary_df.head(top_n))

In [8]:
def check_numeric_summary(df, df_name, cols=None):
    """수치형 컬럼의 기본 기술통계, 왜도, 첨도를 확인"""
    print(f"\n{'='*80}")
    print(f"{df_name}의 수치형 기술통계")
    print(f"{'='*80}")

    if cols is None:
        numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    else:
        numeric_cols = [col for col in cols if col in df.columns]

    if len(numeric_cols) == 0:
        print("수치형 컬럼이 없습니다.")
        return

    summary_df = df[numeric_cols].describe().T
    summary_df["결측치 개수"] = df[numeric_cols].isnull().sum()
    summary_df["왜도"] = df[numeric_cols].skew(numeric_only=True)
    summary_df["첨도"] = df[numeric_cols].kurt(numeric_only=True)

    display(summary_df)

In [9]:
def check_iqr_outlier_summary(df, df_name, cols=None, top_n=30):
    """IQR 기준으로 이상치 후보가 많은 수치형 컬럼을 확인"""
    print(f"\n{'='*80}")
    print(f"{df_name}의 IQR 기준 이상치 후보 확인")
    print(f"{'='*80}")

    if cols is None:
        numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    else:
        numeric_cols = [col for col in cols if col in df.columns]

    if len(numeric_cols) == 0:
        print("수치형 컬럼이 없습니다.")
        return

    rows = []

    for col in numeric_cols:
        temp = df[col].dropna()

        if len(temp) == 0:
            continue

        q1 = temp.quantile(0.25)
        q3 = temp.quantile(0.75)
        iqr = q3 - q1

        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr

        outlier_count = ((temp < lower) | (temp > upper)).sum()

        rows.append({
            "컬럼명": col,
            "확인 행 수": len(temp),
            "결측치 개수": df[col].isnull().sum(),
            "최솟값": temp.min(),
            "Q1": q1,
            "중앙값": temp.median(),
            "Q3": q3,
            "최댓값": temp.max(),
            "IQR": iqr,
            "하한 기준": lower,
            "상한 기준": upper,
            "이상치 후보 개수": outlier_count,
            "이상치 후보 비율(%)": round(outlier_count / len(temp) * 100, 2)
        })

    summary_df = pd.DataFrame(rows)
    summary_df = summary_df.sort_values("이상치 후보 개수", ascending=False)

    display(summary_df.head(top_n))

In [10]:
def check_category_summary(df, df_name, col_name, top_n=10):
    """범주형 컬럼의 값 분포를 확인"""
    print(f"\n{'='*80}")
    print(f"{df_name}의 {col_name} 범주 확인")
    print(f"{'='*80}")

    if col_name not in df.columns:
        print(f"'{col_name}' 컬럼이 존재하지 않습니다.")
        return

    summary_df = df[col_name].value_counts(dropna=False).reset_index()
    summary_df.columns = [col_name, "개수"]
    summary_df["비율(%)"] = (summary_df["개수"] / len(df) * 100).round(2)

    print("전체 행 수:", len(df))
    print(f"{col_name} 고유값 개수(결측 포함):", df[col_name].nunique(dropna=False))
    print()

    display(summary_df.head(top_n))

In [11]:
def check_datetime_parse_summary(df, df_name, col_name):
    """실제 전처리 컬럼을 생성하지 않고, 변환 성공/실패 여부와 날짜 범위만 확인"""
    print(f"\n{'='*80}")
    print(f"{df_name}의 {col_name} 날짜 변환 가능성 확인")
    print(f"{'='*80}")

    if col_name not in df.columns:
        print(f"'{col_name}' 컬럼이 존재하지 않습니다.")
        return

    parsed = pd.to_datetime(df[col_name], errors="coerce")

    summary_df = pd.DataFrame({
        "항목": [
            "전체 행 수", 
            "원본 결측치 수", 
            "날짜 변환 실패 수", 
            "날짜 변환 성공 수", 
            "최소 날짜", 
            "최대 날짜"]
        ,
        "값": [
            len(df),
            df[col_name].isnull().sum(),
            parsed.isnull().sum(),
            parsed.notnull().sum(),
            parsed.min(),
            parsed.max()
        ]
    })

    display(summary_df)

    print("\n연도별 분포")
    display(parsed.dt.year.value_counts(dropna=False).sort_index().reset_index().rename(
        columns={"release_date": "연도", "count": "개수"}
    ))

In [12]:

def check_list_string_summary(df, df_name, col_name, top_n=20):
    """문자열로 저장된 리스트형 컬럼을 파싱해 항목 분포를 확인하는 함수"""
    print(f"\n{'='*80}")
    print(f"{df_name}의 {col_name} 리스트형 문자열 파싱 확인")
    print(f"{'='*80}")

    if col_name not in df.columns:
        print(f"'{col_name}' 컬럼이 존재하지 않습니다.")
        return

    def safe_parse_list(x):
        if pd.isna(x):
            return []
        try:
            parsed = ast.literal_eval(x)
            if isinstance(parsed, list):
                return parsed
            return []
        except Exception:
            return []

    parsed_series = df[col_name].apply(safe_parse_list)

    parse_fail_count = ((df[col_name].notna()) & (parsed_series.apply(len) == 0)).sum()
    list_len = parsed_series.apply(len)

    print("전체 행 수:", len(df))
    print("파싱 실패 또는 빈 리스트 수:", parse_fail_count)
    print("평균 항목 수:", round(list_len.mean(), 2))
    print("최소 항목 수:", list_len.min())
    print("최대 항목 수:", list_len.max())

    exploded = parsed_series.explode()
    summary_df = exploded.value_counts(dropna=False).reset_index()
    summary_df.columns = [col_name, "개수"]
    summary_df["비율(%)"] = (summary_df["개수"] / len(df) * 100).round(2)

    display(summary_df.head(top_n))

In [13]:
def check_many_categories(df, df_name, cols, top_n=10):
    """여러 범주형 컬럼을 같은 형식으로 반복 확인"""
    for col in cols:
        if col in df.columns:
            check_category_summary(df, df_name, col, top_n=top_n)
        else:
            print(f"{df_name}에 '{col}' 컬럼이 없습니다.")

In [14]:
def check_multi_key_duplicates(df, key_cols, df_name, top_n=10):
    """여러 컬럼을 묶어서 기준 키로 볼 때 중복 여부를 확인"""
    print(f"\n{'='*80}")
    print(f"{df_name}의 복합 키 중복 확인: {key_cols}")
    print(f"{'='*80}")

    missing_cols = [col for col in key_cols if col not in df.columns]
    if len(missing_cols) > 0:
        print("존재하지 않는 컬럼:", missing_cols)
        return

    duplicated_count = df.duplicated(subset=key_cols).sum()
    unique_count = df[key_cols].drop_duplicates().shape[0]

    print("전체 행 수:", len(df))
    print("복합 키 고유 조합 수:", unique_count)
    print("복합 키 중복 행 수:", duplicated_count)

    if duplicated_count > 0:
        dup_summary = df.groupby(key_cols).size().reset_index(name="등장 횟수")
        dup_summary = dup_summary[dup_summary["등장 횟수"] > 1].sort_values("등장 횟수", ascending=False)
        display(dup_summary.head(top_n))
    else:
        print("중복 조합이 없습니다.")


In [15]:
def check_timestamp_parse_summary(df, df_name, col_name):
    """Unix timestamp 컬럼을 날짜로 변환할 수 있는지 확인"""
    print(f"\n{'='*80}")
    print(f"{df_name}의 {col_name} 타임스탬프 변환 가능성 확인")
    print(f"{'='*80}")

    if col_name not in df.columns:
        print(f"'{col_name}' 컬럼이 존재하지 않습니다.")
        return

    parsed = pd.to_datetime(df[col_name], unit="s", errors="coerce")

    summary_df = pd.DataFrame({
        "항목": [
            "전체 행 수",
            "원본 결측치 수",
            "타임스탬프 변환 실패 수",
            "타임스탬프 변환 성공 수",
            "최소 날짜",
            "최대 날짜"
        ],
        "값": [
            len(df),
            df[col_name].isnull().sum(),
            parsed.isnull().sum(),
            parsed.notnull().sum(),
            parsed.min(),
            parsed.max()
        ]
    })

    display(summary_df)

    print("\n연도별 분포")
    display(parsed.dt.year.value_counts(dropna=False).sort_index().reset_index().rename(
        columns={"index": "연도", col_name: "연도", "count": "개수"}
    ))

In [16]:
def check_dict_string_summary(df, df_name, col_name, top_n=20):
    """문자열로 저장된 딕셔너리형 컬럼을 파싱해 key 분포를 확인하는 함수"""
    print(f"\n{'='*80}")
    print(f"{df_name}의 {col_name} 딕셔너리형 문자열 파싱 확인")
    print(f"{'='*80}")

    if col_name not in df.columns:
        print(f"'{col_name}' 컬럼이 존재하지 않습니다.")
        return

    def safe_parse_dict_keys(x):
        if pd.isna(x):
            return []
        try:
            parsed = ast.literal_eval(x)
            if isinstance(parsed, dict):
                return list(parsed.keys())
            return []
        except Exception:
            return []

    parsed_series = df[col_name].apply(safe_parse_dict_keys)

    parse_fail_count = ((df[col_name].notna()) & (parsed_series.apply(len) == 0)).sum()
    key_len = parsed_series.apply(len)

    print("전체 행 수:", len(df))
    print("파싱 실패 또는 빈 딕셔너리 수:", parse_fail_count)
    print("평균 key 수:", round(key_len.mean(), 2))
    print("최소 key 수:", key_len.min())
    print("최대 key 수:", key_len.max())

    exploded = parsed_series.explode()
    summary_df = exploded.value_counts(dropna=False).reset_index()
    summary_df.columns = [col_name, "개수"]
    summary_df["비율(%)"] = (summary_df["개수"] / len(df) * 100).round(2)

    display(summary_df.head(top_n))

In [17]:
def check_join_summary(left_df, right_df, left_name, right_name, key="appid", top_n=10):
    """기준 키를 사용해 두 테이블이 얼마나 조인 가능한지 확인"""
    print(f"\n{'='*80}")
    print(f"{left_name} → {right_name} 조인 가능성 확인")
    print(f"{'='*80}")

    if key not in left_df.columns:
        print(f"{left_name}에 '{key}' 컬럼이 없습니다.")
        return

    if key not in right_df.columns:
        print(f"{right_name}에 '{key}' 컬럼이 없습니다.")
        return

    left_keys = set(left_df[key].dropna().astype(str))
    right_keys = set(right_df[key].dropna().astype(str))

    matched_keys = left_keys & right_keys
    unmatched_left_keys = left_keys - right_keys

    result_df = pd.DataFrame({
        "항목": [
            f"{left_name} 고유 {key} 수",
            f"{right_name} 고유 {key} 수",
            "매칭되는 key 수",
            f"{left_name} 기준 미매칭 key 수",
            f"{left_name} 기준 매칭률(%)"
        ],
        "값": [
            len(left_keys),
            len(right_keys),
            len(matched_keys),
            len(unmatched_left_keys),
            round(len(matched_keys) / len(left_keys) * 100, 2) if len(left_keys) > 0 else np.nan
        ]
    })

    display(result_df)

    if len(unmatched_left_keys) > 0:
        print(f"[{left_name}에는 있지만 {right_name}에는 없는 {key} 예시]")
        display(pd.DataFrame({key: list(unmatched_left_keys)[:top_n]}))

# 4. 데이터 점검

## 4-1. `steam_indie_games` 데이터 점검 실행

In [18]:
check_basic_info(games_df, "steam_indie_games")

# 2) 게임 단위 식별자인 appid 중복 확인
check_id_duplicates(games_df, "appid", "steam_indie_games")

# 3) 문자열 컬럼의 공백/빈값 확인
check_string_space_summary(games_df, "steam_indie_games")


steam_indie_games의 기본 정보 / 타입 / 결측치 확인

[전체 요약]


,항목,값
0,행 개수,9692
1,열 개수,14
2,중복 행 개수,0


[컬럼별 요약]


,데이터타입,행 개수,행 비율(%),결측치 개수,결측치 비율(%),고유값 개수
developers,str,9680,99.88,12,0.12,8139
appid,int64,9692,100.00,0,0.00,9692
name,str,9692,100.00,0,0.00,9688
total_reviews,int64,9692,100.00,0,0.00,1427
positive,int64,9692,100.00,0,0.00,1346
release_date,str,9692,100.00,0,0.00,1008
negative,int64,9692,100.00,0,0.00,601
ccu,int64,9692,100.00,0,0.00,294
price,int64,9692,100.00,0,0.00,282
genres,str,9692,100.00,0,0.00,250


[상위 5행]


,appid,owners,positive,negative,price,ccu,genres,release_date,developers,total_reviews,owners_lower,is_f2p,is_early_access,name
0,899770,"20,000,000 .. 50,000,000",88027,22596,3499,5831,"['Action', 'Adventure', 'Indie', 'RPG']",2024-02-21,Eleventh Hour Games,110623,20000000,False,False,Last Epoch
1,251570,"10,000,000 .. 20,000,000",327889,42157,4499,17045,"['Action', 'Adventure', 'Indie', 'RPG', 'Simul...",2024-07-25,The Fun Pimps,370046,10000000,False,False,7 Days to Die
2,1116170,"10,000,000 .. 20,000,000",266,56,1499,3,"['Action', 'Adventure', 'Indie', 'RPG']",2025-04-22,Megame LLC,322,10000000,False,False,CyberCorp
3,1326470,"10,000,000 .. 20,000,000",222495,31051,2999,4450,"['Action', 'Adventure', 'Indie', 'Simulation']",2024-02-22,Endnight Games Ltd,253546,10000000,False,False,Sons Of The Forest
4,2186680,"10,000,000 .. 20,000,000",26360,4445,4999,3582,"['Action', 'Adventure', 'Indie', 'RPG', 'Strat...",2023-12-07,Owlcat Games,30805,10000000,False,False,"Warhammer 40,000: Rogue Trader"



steam_indie_games의 appid 값 중복 확인
전체 행 수: 9692
appid 고유 개수: 9692
중복 appid 개수: 0
중복 값이 없습니다.

steam_indie_games의 문자열 공백/빈값 확인


,컬럼명,문자열 행 수,빈 문자열/공백값 개수,앞뒤 공백 개수,연속 공백 포함 개수
3,developers,9680,0,11,5
4,name,9692,0,7,17
0,owners,9692,0,0,0
1,genres,9692,0,0,0
2,release_date,9692,0,0,0


In [19]:
# 4) 수치형 컬럼 기술통계
# appid는 식별자이므로 수치형 기술통계 해석 대상에서는 제외합니다.
numeric_analysis_cols = [
    "positive",
    "negative",
    "price",
    "ccu",
    "total_reviews",
    "owners_lower"
]

check_numeric_summary(
    games_df,
    "steam_indie_games",
    cols=numeric_analysis_cols
)

# 5) IQR 기준 이상치 후보 확인
# 이상치는 삭제 목적이 아니라, 분포 특성과 극단값 존재 여부를 확인하기 위한 것입니다.
check_iqr_outlier_summary(
    games_df,
    "steam_indie_games",
    cols=numeric_analysis_cols
)



steam_indie_games의 수치형 기술통계


,count,mean,std,min,25%,50%,75%,max,결측치 개수,왜도,첨도
positive,9692.0,696.681903,6750.596063,0.0,15.0,34.0,127.0,327889.0,0,27.696358,996.277080
negative,9692.0,97.027652,1343.304549,0.0,2.0,6.0,22.0,106084.0,0,57.683397,4171.168253
price,9692.0,883.707078,1058.550707,0.0,299.0,599.0,1199.0,19999.0,0,9.899780,165.728969
ccu,9692.0,35.946038,928.321453,0.0,0.0,0.0,1.0,83936.0,0,78.121949,6915.752078
total_reviews,9692.0,793.709554,7662.793747,10.0,18.0,41.0,152.0,370046.0,0,27.808549,990.715919
owners_lower,9692.0,32220.387949,333407.472154,0.0,0.0,0.0,20000.0,20000000.0,0,37.329546,1754.736105



steam_indie_games의 IQR 기준 이상치 후보 확인


,컬럼명,확인 행 수,결측치 개수,최솟값,Q1,중앙값,Q3,최댓값,IQR,하한 기준,상한 기준,이상치 후보 개수,이상치 후보 비율(%)
3,ccu,9692,0,0,0.0,0.0,1.0,83936,1.0,-1.5,2.5,1864,19.23
4,total_reviews,9692,0,10,18.0,41.0,152.0,370046,134.0,-183.0,353.0,1501,15.49
0,positive,9692,0,0,15.0,34.0,127.0,327889,112.0,-153.0,295.0,1500,15.48
1,negative,9692,0,0,2.0,6.0,22.0,106084,20.0,-28.0,52.0,1448,14.94
5,owners_lower,9692,0,0,0.0,0.0,20000.0,20000000,20000.0,-30000.0,50000.0,689,7.11
2,price,9692,0,0,299.0,599.0,1199.0,19999,900.0,-1051.0,2549.0,229,2.36


In [20]:
# 6) 주요 범주형 컬럼 확인
check_category_summary(games_df, "steam_indie_games", "owners", top_n=20)
check_category_summary(games_df, "steam_indie_games", "type")
check_category_summary(games_df, "steam_indie_games", "is_f2p")
check_category_summary(games_df, "steam_indie_games", "is_early_access")



steam_indie_games의 owners 범주 확인
전체 행 수: 9692
owners 고유값 개수(결측 포함): 11



,owners,개수,비율(%)
0,"0 .. 20,000",7228,74.58
1,"20,000 .. 50,000",1209,12.47
2,"50,000 .. 100,000",566,5.84
3,"100,000 .. 200,000",319,3.29
4,"200,000 .. 500,000",237,2.45
5,"500,000 .. 1,000,000",79,0.82
6,"1,000,000 .. 2,000,000",34,0.35
7,"2,000,000 .. 5,000,000",11,0.11
8,"10,000,000 .. 20,000,000",5,0.05
9,"5,000,000 .. 10,000,000",3,0.03



steam_indie_games의 type 범주 확인
'type' 컬럼이 존재하지 않습니다.

steam_indie_games의 is_f2p 범주 확인
전체 행 수: 9692
is_f2p 고유값 개수(결측 포함): 1



,is_f2p,개수,비율(%)
0,False,9692,100.0



steam_indie_games의 is_early_access 범주 확인
전체 행 수: 9692
is_early_access 고유값 개수(결측 포함): 1



,is_early_access,개수,비율(%)
0,False,9692,100.0


In [21]:
# 7) 출시일 변환 가능성 확인
check_datetime_parse_summary(games_df, "steam_indie_games", "release_date")

# 8) 장르 리스트형 문자열 파싱 가능성 확인
check_list_string_summary(games_df, "steam_indie_games", "genres", top_n=30)


steam_indie_games의 release_date 날짜 변환 가능성 확인


,항목,값
0,전체 행 수,9692
1,원본 결측치 수,0
2,날짜 변환 실패 수,0
3,날짜 변환 성공 수,9692
4,최소 날짜,2023-01-01 00:00:00
5,최대 날짜,2025-12-27 00:00:00



연도별 분포


,연도,개수
0,2023,3498
1,2024,4180
2,2025,2014



steam_indie_games의 genres 리스트형 문자열 파싱 확인
전체 행 수: 9692
파싱 실패 또는 빈 리스트 수: 0
평균 항목 수: 3.12
최소 항목 수: 1
최대 항목 수: 10


,genres,개수,비율(%)
0,Indie,9687,99.95
1,Adventure,4807,49.60
2,Casual,4111,42.42
3,Action,4093,42.23
4,Simulation,2503,25.83
5,RPG,2194,22.64
6,Strategy,2005,20.69
7,Sports,341,3.52
8,Racing,301,3.11
9,Massively Multiplayer,124,1.28


## 4-2. `steam_indie_tags` 데이터 점검 실행

In [22]:
# 1) 기본 구조, 타입, 결측치 확인
check_basic_info(tags_df, "steam_indie_tags")

# 2) appid 중복 확인
check_id_duplicates(tags_df, "appid", "steam_indie_tags")

# 3) 문자열 컬럼의 공백/빈값 확인
check_string_space_summary(tags_df, "steam_indie_tags")


steam_indie_tags의 기본 정보 / 타입 / 결측치 확인

[전체 요약]


,항목,값
0,행 개수,9706
1,열 개수,10
2,중복 행 개수,0


[컬럼별 요약]


,데이터타입,행 개수,행 비율(%),결측치 개수,결측치 비율(%),고유값 개수
publisher,str,9673,99.66,33,0.34,7217
developer,str,9694,99.88,12,0.12,8154
appid,int64,9706,100.00,0,0.00,9706
updated_at,str,9706,100.00,0,0.00,9706
tags,str,9706,100.00,0,0.00,9705
name,str,9706,100.00,0,0.00,9701
positive,int64,9706,100.00,0,0.00,1355
negative,int64,9706,100.00,0,0.00,607
price,int64,9706,100.00,0,0.00,295
owners,str,9706,100.00,0,0.00,12


[상위 5행]


,appid,name,developer,publisher,owners,positive,negative,price,tags,updated_at
0,1432860,Sun Haven,Pixel Sprout Studios,Pixel Sprout Studios,"500,000 .. 1,000,000",18523,3933,874,"{""RPG"": 442, ""Magic"": 324, ""Combat"": 287, ""Min...",2026-04-23 13:58:47.087
1,1473350,(the) Gnorp Apologue,Myco,(Myco),"200,000 .. 500,000",7849,294,699,"{""2D"": 117, ""Cute"": 133, ""Idler"": 193, ""Indie""...",2026-04-23 13:58:48.495
2,1993150,轮回修仙路,烟水寒工作室,烟水寒工作室,"200,000 .. 500,000",2355,510,1699,"{""3D"": 359, ""RPG"": 409, ""Indie"": 332, ""Space"":...",2026-04-23 13:58:49.904
3,2527500,MiSide,AIHASTO,"IndieArk, Shochiku (Japan)","1,000,000 .. 2,000,000",108883,2204,1499,"{""2D"": 761, ""3D"": 1392, ""RPG"": 815, ""Cute"": 21...",2026-04-23 13:58:51.311
4,1169040,Necesse,Fair Games ApS,Fair Games ApS,"1,000,000 .. 2,000,000",15988,1083,974,"{""2D"": 188, ""RPG"": 241, ""Co-op"": 237, ""Indie"":...",2026-04-23 13:58:52.692



steam_indie_tags의 appid 값 중복 확인
전체 행 수: 9706
appid 고유 개수: 9706
중복 appid 개수: 0
중복 값이 없습니다.

steam_indie_tags의 문자열 공백/빈값 확인


,컬럼명,문자열 행 수,빈 문자열/공백값 개수,앞뒤 공백 개수,연속 공백 포함 개수
2,publisher,9673,1,77,6
1,developer,9694,0,11,5
0,name,9706,0,8,17
3,owners,9706,0,0,0
4,tags,9706,0,0,0
5,updated_at,9706,0,0,0


In [23]:
# 4) 수치형 컬럼 기술통계
numeric_analysis_cols = [
    "positive",
    "negative",
    "price"
]

check_numeric_summary(
    tags_df,
    "steam_indie_tags",
    cols=numeric_analysis_cols
)

check_iqr_outlier_summary(
    tags_df,
    "steam_indie_tags",
    cols=numeric_analysis_cols
)


steam_indie_tags의 수치형 기술통계


,count,mean,std,min,25%,50%,75%,max,결측치 개수,왜도,첨도
positive,9706.0,809.909953,9279.026061,0.0,15.00,35.0,128.0,473422.0,0,31.188334,1219.290213
negative,9706.0,102.640120,1375.435259,0.0,2.00,6.0,23.0,106084.0,0,54.280882,3797.287638
price,9706.0,873.969297,1064.864328,0.0,340.25,599.0,1099.0,19999.0,0,10.283168,172.438216



steam_indie_tags의 IQR 기준 이상치 후보 확인


,컬럼명,확인 행 수,결측치 개수,최솟값,Q1,중앙값,Q3,최댓값,IQR,하한 기준,상한 기준,이상치 후보 개수,이상치 후보 비율(%)
0,positive,9706,0,0,15.00,35.0,128.0,473422,113.00,-154.500,297.500,1507,15.53
1,negative,9706,0,0,2.00,6.0,23.0,106084,21.00,-29.500,54.500,1432,14.75
2,price,9706,0,0,340.25,599.0,1099.0,19999,758.75,-797.875,2237.125,428,4.41


In [24]:
# 5) 주요 범주형/구조형 컬럼 확인
check_category_summary(tags_df, "steam_indie_tags", "owners", top_n=20)
check_datetime_parse_summary(tags_df, "steam_indie_tags", "updated_at")
check_dict_string_summary(tags_df, "steam_indie_tags", "tags", top_n=30)



steam_indie_tags의 owners 범주 확인
전체 행 수: 9706
owners 고유값 개수(결측 포함): 12



,owners,개수,비율(%)
0,"0 .. 20,000",7233,74.52
1,"20,000 .. 50,000",1210,12.47
2,"50,000 .. 100,000",567,5.84
3,"100,000 .. 200,000",319,3.29
4,"200,000 .. 500,000",239,2.46
5,"500,000 .. 1,000,000",80,0.82
6,"1,000,000 .. 2,000,000",34,0.35
7,"2,000,000 .. 5,000,000",11,0.11
8,"10,000,000 .. 20,000,000",7,0.07
9,"5,000,000 .. 10,000,000",4,0.04



steam_indie_tags의 updated_at 날짜 변환 가능성 확인


,항목,값
0,전체 행 수,9706
1,원본 결측치 수,0
2,날짜 변환 실패 수,0
3,날짜 변환 성공 수,9706
4,최소 날짜,2026-04-23 13:58:47.087000
5,최대 날짜,2026-04-28 15:56:37.822000



연도별 분포


,updated_at,개수
0,2026,9706



steam_indie_tags의 tags 딕셔너리형 문자열 파싱 확인
전체 행 수: 9706
파싱 실패 또는 빈 딕셔너리 수: 0
평균 key 수: 18.23
최소 key 수: 2
최대 key 수: 20


,tags,개수,비율(%)
0,Singleplayer,7462,76.88
1,Indie,5867,60.45
2,Adventure,4642,47.83
3,Casual,4176,43.02
4,Action,4024,41.46
5,2D,3974,40.94
6,3D,3230,33.28
7,Atmospheric,2886,29.73
8,Exploration,2755,28.38
9,Simulation,2486,25.61


## 4-3. `steam_indie_review_summary` 데이터 점검 실행

In [25]:
# 1) 기본 구조, 타입, 결측치 확인
check_basic_info(review_summary_df, "steam_indie_review_summary")

# 2) appid 중복 확인
check_id_duplicates(review_summary_df, "appid", "steam_indie_review_summary")


steam_indie_review_summary의 기본 정보 / 타입 / 결측치 확인

[전체 요약]


,항목,값
0,행 개수,200
1,열 개수,7
2,중복 행 개수,0


[컬럼별 요약]


,데이터타입,행 개수,행 비율(%),결측치 개수,결측치 비율(%),고유값 개수
appid,int64,200,100.0,0,0.0,200
collected_at,int64,200,100.0,0,0.0,200
total_reviews,int64,200,100.0,0,0.0,132
total_positive,int64,200,100.0,0,0.0,124
total_negative,int64,200,100.0,0,0.0,76
review_score_desc,str,200,100.0,0,0.0,15
review_score,int64,200,100.0,0,0.0,7


[상위 5행]


,appid,review_score,review_score_desc,total_positive,total_negative,total_reviews,collected_at
0,2185780,6,Mostly Positive,753,195,948,1777432202
1,3215170,6,Mostly Positive,537,216,753,1777432220
2,1996010,9,Overwhelmingly Positive,6673,126,6799,1777432238
3,2499800,6,Mostly Positive,942,277,1219,1777432358
4,1479140,6,Mostly Positive,801,260,1061,1777432382



steam_indie_review_summary의 appid 값 중복 확인
전체 행 수: 200
appid 고유 개수: 200
중복 appid 개수: 0
중복 값이 없습니다.


In [26]:
# 3) 수치형 컬럼 기술통계
numeric_analysis_cols = [
    "review_score",
    "total_positive",
    "total_negative",
    "total_reviews"
]

check_numeric_summary(
    review_summary_df,
    "steam_indie_review_summary",
    cols=numeric_analysis_cols
)

check_iqr_outlier_summary(
    review_summary_df,
    "steam_indie_review_summary",
    cols=numeric_analysis_cols
)


steam_indie_review_summary의 수치형 기술통계


,count,mean,std,min,25%,50%,75%,max,결측치 개수,왜도,첨도
review_score,200.0,6.005,2.509274,0.0,5.0,7.0,8.00,9.0,0,-1.547803,1.388035
total_positive,200.0,1243.195,7687.769552,0.0,12.0,39.5,244.75,89717.0,0,9.654446,100.937865
total_negative,200.0,90.165,287.846741,0.0,2.0,7.0,39.25,2260.0,0,5.293466,30.210989
total_reviews,200.0,1333.360,7881.049610,0.0,14.0,49.5,310.00,91123.0,0,9.464254,97.497372



steam_indie_review_summary의 IQR 기준 이상치 후보 확인


,컬럼명,확인 행 수,결측치 개수,최솟값,Q1,중앙값,Q3,최댓값,IQR,하한 기준,상한 기준,이상치 후보 개수,이상치 후보 비율(%)
2,total_negative,200,0,0,2.0,7.0,39.25,2260,37.25,-53.875,95.125,34,17.0
3,total_reviews,200,0,0,14.0,49.5,310.00,91123,296.00,-430.000,754.000,28,14.0
1,total_positive,200,0,0,12.0,39.5,244.75,89717,232.75,-337.125,593.875,26,13.0
0,review_score,200,0,0,5.0,7.0,8.00,9,3.00,0.500,12.500,25,12.5


In [27]:
# 4) 주요 범주형/날짜 컬럼 확인
check_category_summary(review_summary_df, "steam_indie_review_summary", "review_score_desc", top_n=20)
check_datetime_parse_summary(review_summary_df, "steam_indie_review_summary", "collected_at")


steam_indie_review_summary의 review_score_desc 범주 확인
전체 행 수: 200
review_score_desc 고유값 개수(결측 포함): 15



,review_score_desc,개수,비율(%)
0,Positive,55,27.5
1,Very Positive,53,26.5
2,Mostly Positive,35,17.5
3,Mixed,24,12.0
4,9 user reviews,7,3.5
5,Overwhelmingly Positive,6,3.0
6,8 user reviews,6,3.0
7,No user reviews,3,1.5
8,5 user reviews,3,1.5
9,6 user reviews,2,1.0



steam_indie_review_summary의 collected_at 날짜 변환 가능성 확인


,항목,값
0,전체 행 수,200
1,원본 결측치 수,0
2,날짜 변환 실패 수,0
3,날짜 변환 성공 수,200
4,최소 날짜,1970-01-01 00:00:01.777432202
5,최대 날짜,1970-01-01 00:00:01.777439465



연도별 분포


,collected_at,개수
0,1970,200


## 4-4. `steam_indie_review_histogram` 데이터 점검 실행

In [28]:
# 1) 기본 구조, 타입, 결측치 확인
check_basic_info(review_histogram_df, "steam_indie_review_histogram")

# 2) appid 중복 확인
# 날짜별 데이터이므로 appid 중복은 자연스러운 구조입니다.
check_id_duplicates(review_histogram_df, "appid", "steam_indie_review_histogram")

# 3) 날짜 단위 중복 확인
check_multi_key_duplicates(review_histogram_df, ["appid", "date"], "steam_indie_review_histogram")
check_multi_key_duplicates(review_histogram_df, ["appid", "date", "data_type"], "steam_indie_review_histogram")



steam_indie_review_histogram의 기본 정보 / 타입 / 결측치 확인

[전체 요약]


,항목,값
0,행 개수,11782
1,열 개수,10
2,중복 행 개수,0


[컬럼별 요약]


,데이터타입,행 개수,행 비율(%),결측치 개수,결측치 비율(%),고유값 개수
date,str,11782,100.0,0,0.0,995
recommendations_up,int64,11782,100.0,0,0.0,321
appid,int64,11782,100.0,0,0.0,200
name,str,11782,100.0,0,0.0,200
release_date,str,11782,100.0,0,0.0,177
hist_start_date,str,11782,100.0,0,0.0,177
hist_end_date,str,11782,100.0,0,0.0,112
recommendations_down,int64,11782,100.0,0,0.0,87
stratum,str,11782,100.0,0,0.0,24
data_type,str,11782,100.0,0,0.0,2


[상위 5행]


...


steam_indie_review_histogram의 appid 값 중복 확인
전체 행 수: 11782
appid 고유 개수: 200
중복 appid 개수: 11582

[중복 상위 값]


,appid,등장 횟수
0,402160,143
1,571740,141
2,1996010,133
3,2629330,132
4,2703850,132
5,444690,127
6,1640630,126
7,2393770,125
8,743130,118
9,2581050,115



steam_indie_review_histogram의 복합 키 중복 확인: ['appid', 'date']
전체 행 수: 11782
복합 키 고유 조합 수: 11461
복합 키 중복 행 수: 321


,appid,date,등장 횟수
137,402160,2026-04-01,2
181,437440,2026-04-01,2
318,444690,2026-04-01,2
442,571740,2026-04-01,2
531,597920,2026-04-01,2
587,695330,2026-04-01,2
703,743130,2026-04-01,2
770,754890,2026-04-01,2
843,837350,2026-01-01,2
878,1014260,2026-02-18,2



steam_indie_review_histogram의 복합 키 중복 확인: ['appid', 'date', 'data_type']
전체 행 수: 11782
복합 키 고유 조합 수: 11782
복합 키 중복 행 수: 0
중복 조합이 없습니다.


In [29]:
# 4) 문자열 공백/빈값 확인
check_string_space_summary(review_histogram_df, "steam_indie_review_histogram")

# 5) 수치형 컬럼 기술통계
numeric_analysis_cols = [
    "recommendations_up",
    "recommendations_down"
]

check_numeric_summary(
    review_histogram_df,
    "steam_indie_review_histogram",
    cols=numeric_analysis_cols
)

check_iqr_outlier_summary(
    review_histogram_df,
    "steam_indie_review_histogram",
    cols=numeric_analysis_cols
)


steam_indie_review_histogram의 문자열 공백/빈값 확인


,컬럼명,문자열 행 수,빈 문자열/공백값 개수,앞뒤 공백 개수,연속 공백 포함 개수
0,name,11782,0,0,0
1,stratum,11782,0,0,0
2,release_date,11782,0,0,0
3,hist_start_date,11782,0,0,0
4,hist_end_date,11782,0,0,0
5,date,11782,0,0,0
6,data_type,11782,0,0,0



steam_indie_review_histogram의 수치형 기술통계


,count,mean,std,min,25%,50%,75%,max,결측치 개수,왜도,첨도
recommendations_up,11782.0,22.107452,350.361376,0.0,0.0,0.0,3.0,29749.0,0,59.085617,4570.554143
recommendations_down,11782.0,1.710745,13.860045,0.0,0.0,0.0,1.0,994.0,0,42.184303,2543.900257



steam_indie_review_histogram의 IQR 기준 이상치 후보 확인


,컬럼명,확인 행 수,결측치 개수,최솟값,Q1,중앙값,Q3,최댓값,IQR,하한 기준,상한 기준,이상치 후보 개수,이상치 후보 비율(%)
0,recommendations_up,11782,0,0,0.0,0.0,3.0,29749,3.0,-4.5,7.5,1770,15.02
1,recommendations_down,11782,0,0,0.0,0.0,1.0,994,1.0,-1.5,2.5,1247,10.58


In [30]:
# 6) 주요 범주형/날짜 컬럼 확인
check_category_summary(review_histogram_df, "steam_indie_review_histogram", "data_type", top_n=20)
check_datetime_parse_summary(review_histogram_df, "steam_indie_review_histogram", "release_date")
check_datetime_parse_summary(review_histogram_df, "steam_indie_review_histogram", "hist_start_date")
check_datetime_parse_summary(review_histogram_df, "steam_indie_review_histogram", "hist_end_date")
check_datetime_parse_summary(review_histogram_df, "steam_indie_review_histogram", "date")


steam_indie_review_histogram의 data_type 범주 확인
전체 행 수: 11782
data_type 고유값 개수(결측 포함): 2



,data_type,개수,비율(%)
0,recent,5976,50.72
1,rollups,5806,49.28



steam_indie_review_histogram의 release_date 날짜 변환 가능성 확인


,항목,값
0,전체 행 수,11782
1,원본 결측치 수,0
2,날짜 변환 실패 수,0
3,날짜 변환 성공 수,11782
4,최소 날짜,2023-01-06 00:00:00
5,최대 날짜,2025-11-08 00:00:00



연도별 분포


...

,hist_end_date,개수
0,2023,75
1,2024,304
2,2025,1647
3,2026,9756



steam_indie_review_histogram의 date 날짜 변환 가능성 확인


,항목,값
0,전체 행 수,11782
1,원본 결측치 수,0
2,날짜 변환 실패 수,0
3,날짜 변환 성공 수,11782
4,최소 날짜,2015-09-01 00:00:00
5,최대 날짜,2026-04-29 00:00:00



연도별 분포


,date,개수
0,2015,4
1,2016,16
2,2017,44
3,2018,33
4,2019,48
5,2020,54
6,2021,79
7,2022,138
8,2023,636
9,2024,1908


## 4-5 `steam_indie_reviews` 데이터 점검 실행

In [31]:
# 1) 기본 구조, 타입, 결측치 확인
# review 본문까지 포함해서 확인합니다.
# 실행 시간이 너무 길면 exclude_cols=["review"]로 바꿔도 됩니다.
check_basic_info(reviews_df, "steam_indie_reviews", exclude_cols=None)

# 2) 리뷰 단위 식별자인 recommendationid 중복 확인
check_id_duplicates(reviews_df, "recommendationid", "steam_indie_reviews")

# 3) appid 중복 확인
# 리뷰 데이터는 한 게임에 여러 리뷰가 있으므로 appid 중복은 자연스러운 구조입니다.
check_id_duplicates(reviews_df, "appid", "steam_indie_reviews")



steam_indie_reviews의 기본 정보 / 타입 / 결측치 확인

[전체 요약]


,항목,값
0,행 개수,236379
1,열 개수,21
2,중복 행 개수,0


[컬럼별 요약]


,데이터타입,행 개수,행 비율(%),결측치 개수,결측치 비율(%),고유값 개수
review,str,235853,99.78,526,0.22,201107
author_playtime_at_review,float64,236374,100.00,5,0.00,8286
recommendationid,int64,236379,100.00,0,0.00,236379
author_last_played,int64,236379,100.00,0,0.00,235686
timestamp_updated,int64,236379,100.00,0,0.00,234807
timestamp_created,int64,236379,100.00,0,0.00,234583
author_steamid,int64,236379,100.00,0,0.00,230361
weighted_vote_score,float64,236379,100.00,0,0.00,22086
author_playtime_forever,int64,236379,100.00,0,0.00,12130
author_num_games_owned,int64,236379,100.00,0,0.00,2994


[상위 5행]


,recommendationid,appid,language,review,timestamp_created,timestamp_updated,voted_up,votes_up,votes_funny,weighted_vote_score,comment_count,steam_purchase,received_for_free,written_during_early_access,author_steamid,author_num_games_owned,author_num_reviews,author_playtime_forever,author_playtime_last_two_weeks,author_playtime_at_review,author_last_played
0,221901691,2185780,schinese,幸存者玩法的游戏，本身属于手游转单机，最大的亮点在于其精致的卡通风格作画，各个可选角色都堪称...,1774689104,1774689104,True,0,0,0.500000,0,True,False,False,76561199700401910,0,55,5078,0,5078.0,1768813677
1,219735714,2185780,schinese,并不推荐，这游戏有些怪外表令人看起来有点恶心（仅个人），并且后期有点难玩，我从第一关到第四关...,1772568518,1772568518,False,0,0,0.500000,0,True,False,False,76561199531512812,72,1,956,0,858.0,1772858411
2,218245394,2185780,schinese,1. 武器重塑：玩法创新的灵魂\n单武器多路线进化：单武器可装 6 个同色 Buff，集齐 ...,1771015143,1771015143,True,0,0,0.500000,0,True,False,False,76561198424126942,20,11,4695,0,4695.0,1692965449
3,213764534,2185780,russian,Единственное что есть в этой игре - это прикол...,1766265572,1766265572,False,2,0,0.540636,0,True,False,False,76561198036737508,0,42,126,0,110.0,1766266077
4,213162568,2185780,english,"Fun fact, if you wanted to 100% this game, at ...",1765592015,1765592015,False,0,0,0.500000,0,True,False,False,76561198245810860,848,114,1539,0,1539.0,1742074336



steam_indie_reviews의 recommendationid 값 중복 확인
전체 행 수: 236379
recommendationid 고유 개수: 236379
중복 recommendationid 개수: 0
중복 값이 없습니다.

steam_indie_reviews의 appid 값 중복 확인
전체 행 수: 236379
appid 고유 개수: 197
중복 appid 개수: 236182

[중복 상위 값]


,appid,등장 횟수
0,1671210,91055
1,3146520,26798
2,571740,24311
3,3244220,17985
4,1811990,8034
5,2820820,6915
6,1996010,6799
7,2703850,6429
8,1747760,2877
9,3205080,2836


In [32]:
# 4) 문자열 컬럼의 공백/빈값 확인
check_string_space_summary(reviews_df, "steam_indie_reviews")


steam_indie_reviews의 문자열 공백/빈값 확인


,컬럼명,문자열 행 수,빈 문자열/공백값 개수,앞뒤 공백 개수,연속 공백 포함 개수
1,review,235853,61,11927,43972
0,language,236379,0,0,0


In [33]:
# 5) 수치형 컬럼 기술통계
numeric_analysis_cols = [
    "votes_up",
    "votes_funny",
    "weighted_vote_score",
    "comment_count",
    "author_num_games_owned",
    "author_num_reviews",
    "author_playtime_forever",
    "author_playtime_last_two_weeks",
    "author_playtime_at_review",
    "author_last_played"
]

check_numeric_summary(
    reviews_df,
    "steam_indie_reviews",
    cols=numeric_analysis_cols
)

check_iqr_outlier_summary(
    reviews_df,
    "steam_indie_reviews",
    cols=numeric_analysis_cols
)


steam_indie_reviews의 수치형 기술통계


,count,mean,std,min,25%,50%,75%,max,결측치 개수,왜도,첨도
votes_up,236379.0,1.567055e+00,2.047584e+01,0.000000e+00,0.000000e+00,0.000000e+00,1.000000e+00,4.705000e+03,0,95.293648,15424.903510
votes_funny,236379.0,1.817015e+04,8.833960e+06,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,4.294967e+09,0,486.188235,236379.000000
weighted_vote_score,236379.0,5.074758e-01,3.737822e-02,6.245356e-02,5.000000e-01,5.000000e-01,5.000000e-01,9.835324e-01,0,4.151516,37.886031
comment_count,236379.0,1.243215e-01,9.554272e-01,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,9.900000e+01,0,39.075197,2615.408925
author_num_games_owned,236379.0,1.486452e+02,6.855281e+02,0.000000e+00,0.000000e+00,0.000000e+00,1.330000e+02,3.488800e+04,0,29.937041,1222.139828
author_num_reviews,236379.0,2.972258e+01,2.845405e+02,1.000000e+00,3.000000e+00,8.000000e+00,2.100000e+01,1.769600e+04,0,55.170601,3345.873327
author_playtime_forever,236379.0,1.912045e+03,7.371719e+03,5.000000e+00,4.510000e+02,1.108000e+03,2.199000e+03,1.819142e+06,0,109.549455,20567.850688
author_playtime_last_two_weeks,236379.0,1.237486e+01,1.806512e+02,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,2.010900e+04,0,60.276009,5329.859229
author_playtime_at_review,236374.0,1.055471e+03,4.033867e+03,1.000000e+00,2.350000e+02,5.690000e+02,1.258000e+03,1.457342e+06,5,215.316313,72984.963957
author_last_played,236379.0,1.745865e+09,3.620472e+07,1.442524e+09,1.739025e+09,1.754344e+09,1.768537e+09,1.777436e+09,0,-2.898118,11.609339



steam_indie_reviews의 IQR 기준 이상치 후보 확인


,컬럼명,확인 행 수,결측치 개수,최솟값,Q1,중앙값,Q3,최댓값,IQR,하한 기준,상한 기준,이상치 후보 개수,이상치 후보 비율(%)
2,weighted_vote_score,236379,0,6.245356e-02,5.000000e-01,5.000000e-01,5.000000e-01,9.835324e-01,0.0,5.000000e-01,5.000000e-01,70029,29.63
5,author_num_reviews,236379,0,1.000000e+00,3.000000e+00,8.000000e+00,2.100000e+01,1.769600e+04,18.0,-2.400000e+01,4.800000e+01,25215,10.67
4,author_num_games_owned,236379,0,0.000000e+00,0.000000e+00,0.000000e+00,1.330000e+02,3.488800e+04,133.0,-1.995000e+02,3.325000e+02,25075,10.61
1,votes_funny,236379,0,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,4.294967e+09,0.0,0.000000e+00,0.000000e+00,21379,9.04
0,votes_up,236379,0,0.000000e+00,0.000000e+00,0.000000e+00,1.000000e+00,4.705000e+03,1.0,-1.500000e+00,2.500000e+00,19934,8.43
9,author_last_played,236379,0,1.442524e+09,1.739025e+09,1.754344e+09,1.768537e+09,1.777436e+09,29512633.0,1.694756e+09,1.812806e+09,17907,7.58
3,comment_count,236379,0,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,9.900000e+01,0.0,0.000000e+00,0.000000e+00,16308,6.90
8,author_playtime_at_review,236374,5,1.000000e+00,2.350000e+02,5.690000e+02,1.258000e+03,1.457342e+06,1023.0,-1.299500e+03,2.792500e+03,15837,6.70
6,author_playtime_forever,236379,0,5.000000e+00,4.510000e+02,1.108000e+03,2.199000e+03,1.819142e+06,1748.0,-2.171000e+03,4.821000e+03,14940,6.32
7,author_playtime_last_two_weeks,236379,0,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,2.010900e+04,0.0,0.000000e+00,0.000000e+00,14235,6.02


In [34]:
# 6) 주요 범주형 컬럼 확인
check_category_summary(reviews_df, "steam_indie_reviews", "language", top_n=20)
check_category_summary(reviews_df, "steam_indie_reviews", "voted_up")
check_category_summary(reviews_df, "steam_indie_reviews", "steam_purchase")
check_category_summary(reviews_df, "steam_indie_reviews", "received_for_free")
check_category_summary(reviews_df, "steam_indie_reviews", "written_during_early_access")


steam_indie_reviews의 language 범주 확인
전체 행 수: 236379
language 고유값 개수(결측 포함): 30



,language,개수,비율(%)
0,english,149233,63.13
1,russian,16989,7.19
2,schinese,14794,6.26
3,spanish,14007,5.93
4,brazilian,7839,3.32
5,german,6335,2.68
6,french,5282,2.23
7,polish,3425,1.45
8,koreana,3247,1.37
9,latam,2718,1.15



steam_indie_reviews의 voted_up 범주 확인
전체 행 수: 236379
voted_up 고유값 개수(결측 포함): 2



,voted_up,개수,비율(%)
0,True,218992,92.64
1,False,17387,7.36



steam_indie_reviews의 steam_purchase 범주 확인
전체 행 수: 236379
steam_purchase 고유값 개수(결측 포함): 1



,steam_purchase,개수,비율(%)
0,True,236379,100.0



steam_indie_reviews의 received_for_free 범주 확인
전체 행 수: 236379
received_for_free 고유값 개수(결측 포함): 2



,received_for_free,개수,비율(%)
0,False,234200,99.08
1,True,2179,0.92



steam_indie_reviews의 written_during_early_access 범주 확인
전체 행 수: 236379
written_during_early_access 고유값 개수(결측 포함): 2



,written_during_early_access,개수,비율(%)
0,False,202752,85.77
1,True,33627,14.23


In [35]:
# 7) 타임스탬프 변환 가능성 확인
check_timestamp_parse_summary(reviews_df, "steam_indie_reviews", "timestamp_created")
check_timestamp_parse_summary(reviews_df, "steam_indie_reviews", "timestamp_updated")
check_timestamp_parse_summary(reviews_df, "steam_indie_reviews", "author_last_played")


steam_indie_reviews의 timestamp_created 타임스탬프 변환 가능성 확인


,항목,값
0,전체 행 수,236379
1,원본 결측치 수,0
2,타임스탬프 변환 실패 수,0
3,타임스탬프 변환 성공 수,236379
4,최소 날짜,2015-09-17 19:32:53
5,최대 날짜,2026-04-29 00:45:13



연도별 분포


,연도,개수
0,2015,37
1,2016,127
2,2017,1310
3,2018,1501
4,2019,2720
5,2020,3964
6,2021,5408
7,2022,3997
8,2023,20412
9,2024,42916



steam_indie_reviews의 timestamp_updated 타임스탬프 변환 가능성 확인


...

,연도,개수
0,2015,30
1,2016,102
2,2017,1253
3,2018,1445
4,2019,2659
5,2020,3889
6,2021,5287
7,2022,3962
8,2023,20070
9,2024,42389



steam_indie_reviews의 author_last_played 타임스탬프 변환 가능성 확인


,항목,값
0,전체 행 수,236379
1,원본 결측치 수,0
2,타임스탬프 변환 실패 수,0
3,타임스탬프 변환 성공 수,236379
4,최소 날짜,2015-09-17 21:08:52
5,최대 날짜,2026-04-29 04:13:56



연도별 분포


,연도,개수
0,2015,12
1,2016,61
2,2017,398
3,2018,570
4,2019,972
5,2020,1539
6,2021,2693
7,2022,2908
8,2023,12802
9,2024,30311


# 5. 선택 데이터 점검 실행

# 6. `appid` 기준 조인 가능성 확인

In [36]:
# steam_indie_games를 본 분석의 기준 테이블로 보고 appid 매칭률을 확인
check_join_summary(tags_df, games_df, "steam_indie_tags", "steam_indie_games", key="appid")
check_join_summary(review_summary_df, games_df, "steam_indie_review_summary", "steam_indie_games", key="appid")
check_join_summary(review_histogram_df, games_df, "steam_indie_review_histogram", "steam_indie_games", key="appid")
check_join_summary(reviews_df, games_df, "steam_indie_reviews", "steam_indie_games", key="appid")


steam_indie_tags → steam_indie_games 조인 가능성 확인


,항목,값
0,steam_indie_tags 고유 appid 수,9706.00
1,steam_indie_games 고유 appid 수,9692.00
2,매칭되는 key 수,9692.00
3,steam_indie_tags 기준 미매칭 key 수,14.00
4,steam_indie_tags 기준 매칭률(%),99.86


[steam_indie_tags에는 있지만 steam_indie_games에는 없는 appid 예시]


,appid
0,2729480
1,2115850
2,2789810
3,2717010
4,3164500
5,1700300
6,2186320
7,1623730
8,1966720
9,1948800



steam_indie_review_summary → steam_indie_games 조인 가능성 확인


,항목,값
0,steam_indie_review_summary 고유 appid 수,200.0
1,steam_indie_games 고유 appid 수,9692.0
2,매칭되는 key 수,200.0
3,steam_indie_review_summary 기준 미매칭 key 수,0.0
4,steam_indie_review_summary 기준 매칭률(%),100.0



steam_indie_review_histogram → steam_indie_games 조인 가능성 확인


,항목,값
0,steam_indie_review_histogram 고유 appid 수,200.0
1,steam_indie_games 고유 appid 수,9692.0
2,매칭되는 key 수,200.0
3,steam_indie_review_histogram 기준 미매칭 key 수,0.0
4,steam_indie_review_histogram 기준 매칭률(%),100.0



steam_indie_reviews → steam_indie_games 조인 가능성 확인


,항목,값
0,steam_indie_reviews 고유 appid 수,197.0
1,steam_indie_games 고유 appid 수,9692.0
2,매칭되는 key 수,197.0
3,steam_indie_reviews 기준 미매칭 key 수,0.0
4,steam_indie_reviews 기준 매칭률(%),100.0


In [37]:
# 반대로 steam_indie_games 기준에서 리뷰/태그 데이터가 얼마나 붙는지도 확인
check_join_summary(games_df, tags_df, "steam_indie_games", "steam_indie_tags", key="appid")
check_join_summary(games_df, review_summary_df, "steam_indie_games", "steam_indie_review_summary", key="appid")
check_join_summary(games_df, review_histogram_df, "steam_indie_games", "steam_indie_review_histogram", key="appid")
check_join_summary(games_df, reviews_df, "steam_indie_games", "steam_indie_reviews", key="appid")


steam_indie_games → steam_indie_tags 조인 가능성 확인


,항목,값
0,steam_indie_games 고유 appid 수,9692.0
1,steam_indie_tags 고유 appid 수,9706.0
2,매칭되는 key 수,9692.0
3,steam_indie_games 기준 미매칭 key 수,0.0
4,steam_indie_games 기준 매칭률(%),100.0



steam_indie_games → steam_indie_review_summary 조인 가능성 확인


,항목,값
0,steam_indie_games 고유 appid 수,9692.00
1,steam_indie_review_summary 고유 appid 수,200.00
2,매칭되는 key 수,200.00
3,steam_indie_games 기준 미매칭 key 수,9492.00
4,steam_indie_games 기준 매칭률(%),2.06


[steam_indie_games에는 있지만 steam_indie_review_summary에는 없는 appid 예시]


...

,appid
0,2319390
1,2810070
2,2347960
3,3344030
4,1100910
5,1549850
6,2891210
7,1562420
8,1744120
9,2832810



steam_indie_games → steam_indie_reviews 조인 가능성 확인


,항목,값
0,steam_indie_games 고유 appid 수,9692.00
1,steam_indie_reviews 고유 appid 수,197.00
2,매칭되는 key 수,197.00
3,steam_indie_games 기준 미매칭 key 수,9495.00
4,steam_indie_games 기준 매칭률(%),2.03


[steam_indie_games에는 있지만 steam_indie_reviews에는 없는 appid 예시]


,appid
0,2319390
1,2810070
2,2347960
3,3344030
4,1100910
5,1549850
6,2891210
7,1562420
8,1744120
9,2832810


## 6-1. 실 조인시 행 수 유지 확인

In [38]:
# 기준 테이블인 steam_indie_games에 태그와 리뷰 요약이 붙을 때 행 수가 유지되는지 확인
join_check_df = games_df.merge(
    tags_df[["appid"]].drop_duplicates(),
    on="appid",
    how="left",
    indicator="tags_merge"
)

join_check_df = join_check_df.merge(
    review_summary_df[["appid"]].drop_duplicates(),
    on="appid",
    how="left",
    indicator="review_summary_merge"
)

print("기준 데이터 shape:", games_df.shape)
print("left join 확인 shape:", join_check_df.shape)

print("\n[tags merge 결과]")
display(join_check_df["tags_merge"].value_counts(dropna=False).reset_index())

print("\n[review summary merge 결과]")
display(join_check_df["review_summary_merge"].value_counts(dropna=False).reset_index())

print("\n[태그가 붙지 않은 게임 예시]")
display(join_check_df[join_check_df["tags_merge"] == "left_only"].head(10))

print("\n[리뷰 요약이 붙지 않은 게임 예시]")
display(join_check_df[join_check_df["review_summary_merge"] == "left_only"].head(10))


기준 데이터 shape: (9692, 14)
left join 확인 shape: (9692, 16)

[tags merge 결과]


,tags_merge,count
0,both,9692
1,left_only,0
2,right_only,0



[review summary merge 결과]


,review_summary_merge,count
0,left_only,9492
1,both,200
2,right_only,0



[태그가 붙지 않은 게임 예시]


,appid,owners,positive,negative,price,ccu,genres,release_date,developers,total_reviews,owners_lower,is_f2p,is_early_access,name,tags_merge,review_summary_merge



[리뷰 요약이 붙지 않은 게임 예시]


,appid,owners,positive,negative,price,ccu,genres,release_date,developers,total_reviews,owners_lower,is_f2p,is_early_access,name,tags_merge,review_summary_merge
0,899770,"20,000,000 .. 50,000,000",88027,22596,3499,5831,"['Action', 'Adventure', 'Indie', 'RPG']",2024-02-21,Eleventh Hour Games,110623,20000000,False,False,Last Epoch,both,left_only
1,251570,"10,000,000 .. 20,000,000",327889,42157,4499,17045,"['Action', 'Adventure', 'Indie', 'RPG', 'Simul...",2024-07-25,The Fun Pimps,370046,10000000,False,False,7 Days to Die,both,left_only
2,1116170,"10,000,000 .. 20,000,000",266,56,1499,3,"['Action', 'Adventure', 'Indie', 'RPG']",2025-04-22,Megame LLC,322,10000000,False,False,CyberCorp,both,left_only
3,1326470,"10,000,000 .. 20,000,000",222495,31051,2999,4450,"['Action', 'Adventure', 'Indie', 'Simulation']",2024-02-22,Endnight Games Ltd,253546,10000000,False,False,Sons Of The Forest,both,left_only
4,2186680,"10,000,000 .. 20,000,000",26360,4445,4999,3582,"['Action', 'Adventure', 'Indie', 'RPG', 'Strat...",2023-12-07,Owlcat Games,30805,10000000,False,False,"Warhammer 40,000: Rogue Trader",both,left_only
5,526870,"10,000,000 .. 20,000,000",225479,6585,3999,12596,"['Adventure', 'Indie', 'Simulation', 'Strategy']",2024-09-10,Coffee Stain Studios,232064,10000000,False,False,Satisfactory,both,left_only
6,2881650,"5,000,000 .. 10,000,000",137483,7608,799,722,"['Action', 'Adventure', 'Indie']",2024-04-01,"Zorro, Wilnyl, Philip, thePetHen, Skog",145091,5000000,False,False,Content Warning,both,left_only
7,513710,"5,000,000 .. 10,000,000",83704,28411,1799,8535,"['Action', 'Adventure', 'Indie', 'Massively Mu...",2025-06-17,Gamepires,112115,5000000,False,False,SCUM,both,left_only
8,1144200,"5,000,000 .. 10,000,000",210654,25542,4999,4296,"['Action', 'Adventure', 'Indie']",2023-12-13,VOID Interactive,236196,5000000,False,False,Ready or Not,both,left_only
9,1145350,"2,000,000 .. 5,000,000",60844,3288,2399,1838,"['Action', 'Indie', 'RPG']",2025-09-25,Supergiant Games,64132,2000000,False,False,Hades II,both,left_only


# 보고서

## 점검 대상 데이터

| 데이터명 | 행 수 | 열 수 | 역할 |
|---|---:|---:|---|
| `steam_indie_games` | 9,593 | 15 | 본 분석의 기준이 되는 Steam 인디게임 메타데이터 |
| `steam_indie_tags` | 9,706 | 10 | 게임별 Steam 태그 및 태그 기반 메타정보 |
| `steam_indie_review_summary` | 200 | 7 | 표본 게임의 전체 리뷰 수, 긍정/부정 수, Steam 평점 요약 |
| `steam_indie_review_histogram` | 11,782 | 10 | 표본 게임의 날짜별 긍정/부정 리뷰 변화 |
| `steam_indie_reviews` | 236,379 | 21 | 표본 게임의 개별 리뷰 본문, 추천 여부, 작성 시점, 플레이타임 |

| 데이터명 | 전체 행 수 | 전체 열 수 | 완전 중복 행 수 | 주요 키 | 주요 키 고유값 수 | 주요 키 중복 수 |
|---|---:|---:|---:|---|---:|---:|
| `steam_indie_games` | 9,593 | 15 | 0 | `appid` | 9,593 | 0 |
| `steam_indie_tags` | 9,706 | 10 | 0 | `appid` | 9,706 | 0 |
| `steam_indie_review_summary` | 200 | 7 | 0 | `appid` | 200 | 0 |
| `steam_indie_review_histogram` | 11,782 | 10 | 0 | `appid` | 200 | 11,582 |
| `steam_indie_reviews` | 236,379 | 21 | 0 | `recommendationid` | 236,379 | 0 |

# 확인한 내용

## `steam_indie_games` 점검 결과
본 분석의 기준 테이블\
게임 단위 분석에서 가장 먼저 기준으로 삼을 수 있는 데이터이며, 각 행은 게임 1개를 의미

### 기본 구조

| 항목 | 결과 |
|---|---:|
| 행 수 | 9,593 |
| 열 수 | 15 |
| 완전 중복 행 수 | 0 |
| `appid` 고유값 수 | 9,593 |
| `appid` 중복 수 | 0 |

`appid` 중복이 없으므로, 게임 단위 기준 테이블로 사용하기에 적절하다.


### 결측치

| 컬럼 | 결측치 수 | 결측 비율 |
|---|---:|---:|
| `developers` | 12 | 0.13% |

결측치는 `developers`에서만 소량 확인\
게임 단위 성과 분석에는 큰 영향이 없을 가능성이 높지만, 개발사 기준 집계나 개발사명 기반 필터링을 할 경우에는 확인이 필요

### 문자열 공백

| 컬럼 | 빈 문자열/공백값 | 앞뒤 공백 | 연속 공백 | 줄바꿈 포함 |
|---|---:|---:|---:|---:|
| `developers` | 0 | 11 | 5 | 0 |
| `name` | 0 | 7 | 17 | 0 |

게임명과 개발사명에 일부 공백 문제 발생\
따라서 이름 기준 집계나 조인을 진행하기 전에는 `str.strip()`으로 앞뒤 공백을 제거하고, 필요하면 연속 공백도 정리하는 것이 필요

### 수치형 변수 분포

| 컬럼 | 중앙값 | 최댓값 | 0값 수 | IQR 기준 이상치 후보 수 |
|---|---:|---:|---:|---:|
| `positive` | 35 | 327,889 | 2 | 1,490 |
| `negative` | 6 | 106,084 | 1,345 | 1,419 |
| `total_reviews` | 41 | 370,046 | 0 | 1,475 |
| `price` | 599 | 19,999 | 190 | 228 |
| `ccu` | 0 | 83,936 | 6,090 | 1,864 |
| `owners_lower` | 0 | 20,000,000 | 7,129 | 689 |

리뷰 수, 소유자 수, 동시접속자 수는 우측 치우침이 강하다.\
이는 Steam 인디게임 시장에서 소수의 대형 성공작과 다수의 소규모 게임이 함께 존재하기 때문으로 볼 수 있다.\
따라서 단순 삭제 대상이 아니라, 흥행 규모 차이를 보여주는 주요 특성으로 해석하는 것이 적절해 보인다.

### 출시일과 장르

| 항목 | 결과 |
|---|---|
| 출시일 변환 실패 수 | 0 |
| 최소 출시일 | 2023-01-01 |
| 최대 출시일 | 2025-12-27 |
| `genres` 파싱 실패 수 | 0 |
| `genres` 평균 항목 수 | 3.12개 |

`release_date`는 날짜형으로 변환 가능하고, `genres`도 문자열 리스트 형태에서 정상적으로 파싱된다.\

### 주요 장르 분포

| 장르 | 개수 | 비율 |
|---|---:|---:|
| Indie | 9,588 | 99.95% |
| Adventure | 4,759 | 49.61% |
| Action | 4,050 | 42.22% |
| Casual | 4,044 | 42.16% |
| Simulation | 2,472 | 25.77% |
| RPG | 2,176 | 22.68% |
| Strategy | 1,987 | 20.71% |

## `steam_indie_tags` 점검 결과
`steam_indie_tags`는 Steam 태그 기반의 세부 특성을 확인하기 위한 데이터\

### 기본 구조

| 항목 | 결과 |
|---|---:|
| 행 수 | 9,706 |
| 열 수 | 10 |
| 완전 중복 행 수 | 0 |
| `appid` 고유값 수 | 9,706 |
| `appid` 중복 수 | 0 |

태그 데이터 역시 `appid` 중복이 없으므로 게임 1개당 1행 구조

### 결측치

| 컬럼 | 결측치 수 | 결측 비율 |
|---|---:|---:|
| `publisher` | 33 | 0.34% |
| `developer` | 12 | 0.12% |

결측 비율은 낮다.\
다만 배급사나 개발사 기준 분석을 할 경우에는 `publisher`, `developer` 결측 확인 필요

### 문자열 공백

| 컬럼 | 빈 문자열/공백값 | 앞뒤 공백 | 연속 공백 | 줄바꿈 포함 |
|---|---:|---:|---:|---:|
| `publisher` | 1 | 77 | 6 | 0 |
| `developer` | 0 | 11 | 5 | 0 |
| `name` | 0 | 8 | 17 | 0 |

`publisher`에서 빈 문자열/공백값 1개가 확인되므로, 전처리 시 빈 문자열을 결측으로 통일하는 것이 좋다.

### 태그 구조

| 항목 | 결과 |
|---|---:|
| `tags` 파싱 실패 수 | 0 |
| 게임당 평균 태그 수 | 18.23개 |
| 최소 태그 수 | 2개 |
| 최대 태그 수 | 20개 |

`tags`는 딕셔너리 형태의 문자열로 저장되어 있으며, 파싱 실패 없이 변환 가능하다.

### 주요 태그 분포

| 태그 | 등장 게임 수 | 비율 |
|---|---:|---:|
| Singleplayer | 7,462 | 76.88% |
| Indie | 5,867 | 60.45% |
| Adventure | 4,642 | 47.83% |
| Casual | 4,176 | 43.02% |
| Action | 4,024 | 41.46% |
| 2D | 3,974 | 40.94% |
| 3D | 3,230 | 33.28% |
| Atmospheric | 2,886 | 29.73% |
| Exploration | 2,755 | 28.38% |
| Simulation | 2,486 | 25.61% |


## `steam_indie_review_summary` 점검 결과
`steam_indie_review_summary`는 표본 200개 게임의 리뷰 요약 데이터\
전체 긍정/부정 리뷰 수, 전체 리뷰 수, Steam 평점 요약을 게임 단위로 확인 가능

### 기본 구조

| 항목 | 결과 |
|---|---:|
| 행 수 | 200 |
| 열 수 | 7 |
| 완전 중복 행 수 | 0 |
| `appid` 고유값 수 | 200 |
| `appid` 중복 수 | 0 |
| 결측치 수 | 0 |

게임 1개당 1행 구조이며 결측이 없다.

### 주요 수치형 변수

| 컬럼 | 중앙값 | 평균 | 최댓값 | 0값 수 |
|---|---:|---:|---:|---:|
| `total_positive` | 39.5 | 1,243.19 | 89,717 | 4 |
| `total_negative` | 7 | 90.17 | 2,260 | 21 |
| `total_reviews` | 49.5 | 1,333.36 | 91,123 | 3 |
| `review_score` | 7 | 6.00 | 9 | 25 |

표본 200개 중 일부 게임은 리뷰 수가 거의 없거나 없는 상태\
`total_reviews`가 0인 게임이 3개 있으므로, 긍정률 계산 시 0으로 나누는 문제가 생기지 않도록 처리해야 한다.

### Steam 평점 요약 분포

| 평점 설명 | 개수 | 비율 |
|---|---:|---:|
| Positive | 55 | 27.5% |
| Very Positive | 53 | 26.5% |
| Mostly Positive | 35 | 17.5% |
| Mixed | 24 | 12.0% |
| Overwhelmingly Positive | 6 | 3.0% |
| Mostly Negative | 2 | 1.0% |
| No user reviews | 3 | 1.5% |
| 9 user reviews | 7 | 3.5% |
| 8 user reviews | 6 | 3.0% |

`review_score_desc`에는 일반적인 Steam 평점뿐 아니라 `9 user reviews`, `No user reviews`처럼 리뷰 수가 적은 게임의 설명도 함께 들어 있다.

## `steam_indie_review_histogram` 점검 결과
`steam_indie_review_histogram`은 표본 게임의 날짜별 리뷰 변화 데이터

### 기본 구조

| 항목 | 결과 |
|---|---:|
| 행 수 | 11,782 |
| 열 수 | 10 |
| 완전 중복 행 수 | 0 |
| 고유 `appid` 수 | 200 |
| 결측치 수 | 0 |

하나의 게임에 여러 날짜의 리뷰 변화가 붙어 있으므로, `appid` 중복은 정상적인 구조

### 중복 키 확인

| 기준 키 | 중복 행 수 | 해석 |
|---|---:|---|
| `appid + date` | 321 | 같은 날짜에 서로 다른 `data_type`이 존재할 수 있음 |
| `appid + date + data_type` | 0 | 날짜별 리뷰 변화 데이터의 고유 키로 사용 가능 |

따라서 이 데이터의 행 식별 기준은 `appid + date + data_type`으로 보는 것이 적절

### 날짜 변환

| 컬럼 | 변환 실패 수 | 최소 날짜 | 최대 날짜 |
|---|---:|---|---|
| `release_date` | 0 | 2023-01-06 | 2025-11-08 |
| `hist_start_date` | 0 | 2015-09-17 | 2025-06-04 |
| `hist_end_date` | 0 | 2023-09-24 | 2026-04-29 |
| `date` | 0 | 2015-09-01 | 2026-04-29 |

날짜 컬럼은 모두 정상적으로 변환 가능\
다만, `date`와 `hist_start_date`는 실제 게임 출시일보다 오래된 날짜가 포함될 수 있으므로, 초기 반응 분석에서는 반드시 `release_date`를 기준으로 D+N 기간을 다시 계산해야 한다.

### 리뷰 변화 수치

| 컬럼 | 중앙값 | 최댓값 | 0값 수 | IQR 기준 이상치 후보 수 |
|---|---:|---:|---:|---:|
| `recommendations_up` | 0 | 29,749 | 6,019 | 1,770 |
| `recommendations_down` | 0 | 994 | 8,551 | 1,247 |

날짜별 리뷰 변화는 0이 매우 많고, 일부 날짜에 리뷰가 몰리는 구조\
이는 출시일, 할인, 업데이트, 외부 화제성 등에 의해 특정 날짜에 리뷰가 집중될 수 있기 때문

따라서 이 데이터는 단순 평균보다 누적합, 기간합, 출시 후 N일 이내 변화량 같은 방식으로 사용하는 것이 적절

## `steam_indie_reviews` 점검 결과
`steam_indie_reviews`는 개별 리뷰 본문 데이터\
부정 리뷰 원인 분류, 긍정/부정 리뷰 비교, 플레이타임 기반 소비 방식 진단에 활용 가능

### 기본 구조

| 항목 | 결과 |
|---|---:|
| 행 수 | 236,379 |
| 열 수 | 21 |
| 완전 중복 행 수 | 0 |
| 고유 `appid` 수 | 197 |
| `recommendationid` 고유값 수 | 236,379 |
| `recommendationid` 중복 수 | 0 |

개별 리뷰 데이터는 `recommendationid`를 기준으로 고유 행을 식별할 수 있다.\
따라서 리뷰 단위 분석에서는 `recommendationid`를 기본 키로 보면 된다.

### 결측치

| 컬럼 | 결측치 수 | 결측 비율 |
|---|---:|---:|
| `review` | 526 | 0.22% |
| `author_playtime_at_review` | 5 | 0.00% |

리뷰 본문 결측이 526개 존재한다.\
텍스트 분석에서는 `review` 결측과 빈 문자열을 제외하거나 별도 라벨로 처리해야 한다.

### 문자열 공백 및 줄바꿈

| 컬럼 | 빈 문자열/공백값 | 앞뒤 공백 | 연속 공백 | 줄바꿈 포함 |
|---|---:|---:|---:|---:|
| `review` | 61 | 11,927 | 43,972 | 52,710 |

리뷰 본문에는 줄바꿈과 연속 공백이 많이 포함되어 있다.\
LLM 분석이나 키워드 분석을 진행하기 전에는 줄바꿈, 탭, 연속 공백 정리가 필요하다.

### 리뷰 언어 분포

| 언어 | 개수 | 비율 |
|---|---:|---:|
| english | 149,233 | 63.13% |
| russian | 16,989 | 7.19% |
| schinese | 14,794 | 6.26% |
| spanish | 14,007 | 5.93% |
| brazilian | 7,839 | 3.32% |
| german | 6,335 | 2.68% |
| french | 5,282 | 2.23% |
| koreana | 3,247 | 1.37% |

리뷰 본문은 다국어 데이터다.

### 추천 여부 및 구매 여부

| 컬럼 | 값 | 개수 | 비율 |
|---|---|---:|---:|
| `voted_up` | True | 218,992 | 92.64% |
| `voted_up` | False | 17,387 | 7.36% |
| `steam_purchase` | True | 236,379 | 100.00% |
| `received_for_free` | False | 234,200 | 99.08% |
| `received_for_free` | True | 2,179 | 0.92% |

개별 리뷰 데이터는 긍정 리뷰 비중이 매우 높다.\
부정 리뷰 원인 분석을 하려면 `voted_up = False`인 리뷰를 별도로 추출해서 보는 것이 적절하다.

또한 모든 리뷰가 `steam_purchase = True`로 확인되므로, 구매 여부 기준의 신뢰도는 비교적 안정적이다.

# 조인 가능성 확인

| 기준 테이블 | 대상 테이블 | 기준 appid 수 | 매칭 appid 수 | 미매칭 appid 수 | 매칭률 | 대상에만 있는 appid 수 |
|---|---|---:|---:|---:|---:|---:|
| `steam_indie_games` | `steam_indie_tags` | 9,593 | 9,593 | 0 | 100.00% | 113 |
| `steam_indie_tags` | `steam_indie_games` | 9,706 | 9,593 | 113 | 98.84% | 0 |
| `steam_indie_games` | `steam_indie_review_summary` | 9,593 | 197 | 9,396 | 2.05% | 3 |
| `steam_indie_review_summary` | `steam_indie_games` | 200 | 197 | 3 | 98.50% | 9,396 |
| `steam_indie_games` | `steam_indie_review_histogram` | 9,593 | 197 | 9,396 | 2.05% | 3 |
| `steam_indie_review_histogram` | `steam_indie_games` | 200 | 197 | 3 | 98.50% | 9,396 |
| `steam_indie_games` | `steam_indie_reviews` | 9,593 | 194 | 9,399 | 2.02% | 3 |
| `steam_indie_reviews` | `steam_indie_games` | 197 | 194 | 3 | 98.48% | 9,399 |


## 조인 구조 정리
| 분석 목적 | 기준 테이블 | 조인 대상 | 조인 방식 |
|---|---|---|---|
| 게임 속성 분석 | `steam_indie_games` | `steam_indie_tags` | `appid` 기준 1:1 조인 |
| 표본 게임 리뷰 요약 분석 | `steam_indie_review_summary` | `steam_indie_games`, `steam_indie_tags` | 표본 200개 기준 조인 |
| 날짜별 리뷰 추세 분석 | `steam_indie_review_histogram` | `steam_indie_games` | `appid` 기준 1:N 구조 유지 |
| 개별 리뷰 텍스트 분석 | `steam_indie_reviews` | `steam_indie_games` | `appid` 기준 1:N 구조 유지 |


## 본 분석 전 필요한 전처리 후보

| 대상 데이터 | 전처리 항목 | 이유 |
|---|---|---|
| 전체 문자열 컬럼 | 앞뒤 공백 제거 | 게임명, 개발사명, 배급사명 기준 집계 오류 방지 |
| 전체 문자열 컬럼 | 빈 문자열/공백값을 결측으로 통일 | 실제 결측과 빈 문자열을 동일하게 처리하기 위함 |
| `steam_indie_games` | `release_date` 날짜형 변환 | 출시 연도/월, 출시 후 기간 계산에 필요 |
| `steam_indie_games` | `genres` 리스트 파싱 | 장르별 분석에 필요 |
| `steam_indie_tags` | `tags` 딕셔너리 파싱 | 태그별 분석에 필요 |
| `steam_indie_review_summary` | `total_reviews = 0` 처리 | 긍정률 계산 시 0으로 나누는 문제 방지 |
| `steam_indie_review_histogram` | `date`, `release_date` 날짜형 변환 | D7, D30 같은 출시 후 기간 계산에 필요 |
| `steam_indie_review_histogram` | `appid + date + data_type` 기준 키 확인 | 날짜별 리뷰 변화 데이터의 고유 행 기준 설정 |
| `steam_indie_reviews` | `timestamp_created`, `timestamp_updated` 변환 | 리뷰 작성 시점 기반 분석에 필요 |
| `steam_indie_reviews` | `review` 결측/빈 문자열 처리 | 텍스트 분석 오류 방지 |
| `steam_indie_reviews` | 리뷰 본문 줄바꿈/연속 공백 정리 | 키워드 분석, LLM 분석 전 텍스트 정돈 필요 |
| 조인 전 | 리뷰 표본에만 존재하는 3개 `appid` 확인 | 메타데이터 누락 여부 판단 필요 |